# CA-APSRG Retinal Vessel Segmentation — Google Colab Notebook

Notebook ini menjalankan project `ca_apsrg_retinal_project` di Google Colab, mulai dari persiapan repository, dataset, konversi PNG, pembuatan manifest, menjalankan eksperimen 1–6, mengembalikan hasil utama ke Experiment 3, dan membuat ZIP hasil eksperimen.

**Catatan penting:**
- Dataset penuh tidak disertakan di notebook ini.
- Siapkan dataset DRIVE, STARE, dan CHASEDB1 di Google Drive atau upload sebagai ZIP.
- Untuk hasil utama Streamlit Cloud, notebook ini akan menyalin **Experiment 3: Balanced CA-APSRG** ke:
  - `outputs/experiments/`
  - `outputs/analysis/`


## 1. Pilih Runtime

Di Colab, pilih:

`Runtime` → `Change runtime type` → `Python 3`

GPU **tidak wajib** untuk project ini karena pipeline CA-APSRG saat ini berbasis NumPy, OpenCV, scikit-image, dan PIL. CPU Colab sudah cukup untuk menjalankan eksperimen dataset kecil-menengah.


In [ ]:
from pathlib import Path
import os
import sys
import shutil
import subprocess
import json
import textwrap
import time

print("Python executable:", sys.executable)
print("Current directory:", Path.cwd())


## 2. Mount Google Drive

Gunakan Google Drive untuk menyimpan dataset dan hasil ZIP agar tidak hilang ketika sesi Colab berakhir.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Clone Repository dan Install Dependency

Ubah `REPO_URL` jika repository dipindahkan atau menggunakan branch lain.


In [ ]:
REPO_URL = "https://github.com/nicolauseuclides512/ca_apsrg_retinal_project.git"
PROJECT_DIR = Path("/content/ca_apsrg_retinal_project")

if PROJECT_DIR.exists():
    print(f"Project already exists: {PROJECT_DIR}")
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", Path.cwd())
print("Top-level files:")
for p in sorted(PROJECT_DIR.iterdir()):
    print("-", p.name)


In [ ]:
# Install dependency project
# Streamlit tidak wajib untuk eksperimen batch, tetapi tetap diinstall agar app.py bisa dites jika diperlukan.
!pip install -q -r requirements.txt
!pip install -q streamlit

print("Dependency installation finished.")


## 4. Persiapan Dataset

Project mengharapkan dataset mentah berada di:

```text
data/raw/Retinal Vessel/
├── DRIVE/
├── STARE/
└── CHASEDB1/
```

Ada dua cara:

1. **Dataset sudah diekstrak di Google Drive**  
   Contoh: `/content/drive/MyDrive/Retinal Vessel/`

2. **Dataset masih dalam ZIP di Google Drive**  
   Contoh: `/content/drive/MyDrive/Retinal Vessel.zip`

Atur variabel di cell berikut sesuai lokasi dataset Bapak.


In [ ]:
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw" / "Retinal Vessel"

# Opsi A: dataset sudah berupa folder di Google Drive
DATASET_FOLDER_ON_DRIVE = Path("/content/drive/MyDrive/Retinal Vessel")

# Opsi B: dataset masih berupa archive/zip di Google Drive
DATASET_ARCHIVE_ON_DRIVE = Path("/content/drive/MyDrive/Retinal Vessel.zip")

# Pilih mode:
# "folder"  = copy dari folder Google Drive
# "archive" = extract dari ZIP/archive Google Drive
# "skip"    = jika dataset sudah ada di data/raw/Retinal Vessel
DATASET_MODE = "folder"  # ganti menjadi "archive" atau "skip" jika diperlukan

print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("DATASET_MODE:", DATASET_MODE)


In [ ]:
def print_tree(root: Path, max_depth: int = 3, max_items: int = 80) -> None:
    root = Path(root)
    if not root.exists():
        print(f"[missing] {root}")
        return

    root_depth = len(root.parts)
    count = 0
    for path in sorted(root.rglob("*")):
        depth = len(path.parts) - root_depth
        if depth > max_depth:
            continue
        indent = "  " * depth
        print(f"{indent}- {path.name}")
        count += 1
        if count >= max_items:
            print("...")
            break


def reset_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


if DATASET_MODE == "folder":
    if not DATASET_FOLDER_ON_DRIVE.exists():
        raise FileNotFoundError(f"Dataset folder not found: {DATASET_FOLDER_ON_DRIVE}")

    reset_dir(RAW_DATA_DIR)
    # Copy isi folder "Retinal Vessel" ke data/raw/Retinal Vessel
    for item in DATASET_FOLDER_ON_DRIVE.iterdir():
        dest = RAW_DATA_DIR / item.name
        if item.is_dir():
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)

elif DATASET_MODE == "archive":
    if not DATASET_ARCHIVE_ON_DRIVE.exists():
        raise FileNotFoundError(f"Dataset archive not found: {DATASET_ARCHIVE_ON_DRIVE}")

    reset_dir(RAW_DATA_DIR)
    temp_extract = PROJECT_DIR / "data" / "raw" / "_temp_extract"
    reset_dir(temp_extract)
    shutil.unpack_archive(str(DATASET_ARCHIVE_ON_DRIVE), str(temp_extract))

    # Jika archive berisi satu folder bernama Retinal Vessel, ambil isinya.
    candidates = [p for p in temp_extract.iterdir()]
    if len(candidates) == 1 and candidates[0].is_dir():
        source_root = candidates[0]
    else:
        source_root = temp_extract

    for item in source_root.iterdir():
        dest = RAW_DATA_DIR / item.name
        if item.is_dir():
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)

    shutil.rmtree(temp_extract)

elif DATASET_MODE == "skip":
    print("Skip dataset copy/extract. Using existing raw dataset.")

else:
    raise ValueError("DATASET_MODE must be one of: folder, archive, skip")

print("Dataset tree preview:")
print_tree(RAW_DATA_DIR, max_depth=3, max_items=100)


## 5. Konversi Dataset ke PNG dan Buat Manifest

Jalankan bagian ini ketika:
- dataset baru dipasang di Colab,
- folder `data/working_png/` belum ada,
- atau `data/manifests/manifest.csv` belum ada.

Jika sudah pernah dibuat dan tidak berubah, cell ini tetap aman dijalankan ulang.


In [ ]:
def run_cmd(cmd, cwd: Path = PROJECT_DIR, check: bool = True) -> subprocess.CompletedProcess:
    print("\n$", " ".join(map(str, cmd)))
    start = time.time()
    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    elapsed = time.time() - start
    print(result.stdout)
    print(f"[finished in {elapsed:.1f}s]")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


run_cmd([
    sys.executable,
    "scripts/00_convert_all_to_png.py",
    "--input-root", "data/raw/Retinal Vessel",
    "--output-root", "data/working_png",
    "--copy-png",
    "--binarize-masks",
])

run_cmd([
    sys.executable,
    "scripts/01_build_manifest.py",
    "--working-root", "data/working_png",
    "--output", "data/manifests/manifest.csv",
])

manifest_path = PROJECT_DIR / "data" / "manifests" / "manifest.csv"
print("Manifest exists:", manifest_path.exists(), manifest_path)


In [ ]:
import pandas as pd

manifest = pd.read_csv(PROJECT_DIR / "data/manifests/manifest.csv")
print(manifest.head())
print("\nDataset count:")
print(manifest.groupby("dataset").size())


## 6. Uji Satu Gambar

Cell ini opsional, tetapi baik untuk memastikan pipeline berjalan sebelum menjalankan batch semua eksperimen.


In [ ]:
# Uji satu gambar dari manifest row pertama.
run_cmd([
    sys.executable,
    "scripts/02_run_single_image.py",
    "--manifest", "data/manifests/manifest.csv",
    "--row-index", "0",
    "--output-dir", "outputs/single_test_colab",
])


## 7. Definisi Konfigurasi Eksperimen 1–6

Notebook ini membuat file konfigurasi terpisah di:

```text
configs/experiments/
```

Eksperimen:
1. `exp1_recall_oriented` — closing + fill holes, cenderung menaikkan recall.
2. `exp2_precision_oriented` — opening + small component removal besar, cenderung menaikkan precision.
3. `exp3_balanced_main` — konfigurasi utama yang direkomendasikan.
4. `exp4_static_morphology` — small component threshold statis 18/18/18.
5. `exp5_no_skeleton_guard` — seperti Exp 3 tetapi skeleton guard dimatikan.
6. `exp6_no_small_component` — small component removal dimatikan, untuk ablation.


In [ ]:
import yaml
from copy import deepcopy

CONFIG_DIR = PROJECT_DIR / "configs"
EXPERIMENT_CONFIG_DIR = CONFIG_DIR / "experiments"
EXPERIMENT_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

BASE_CONFIG_PATH = CONFIG_DIR / "default.yaml"

def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

def save_yaml(data: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

base_config = load_yaml(BASE_CONFIG_PATH)

COMMON_ZERO_KERNELS = {
    "closing_kernel_low_density": 0,
    "closing_kernel_normal": 0,
    "closing_kernel_high_density": 0,
    "opening_kernel_low_density": 0,
    "opening_kernel_normal": 0,
    "opening_kernel_high_density": 0,
    "skeleton_restore_radius": 1,
    "skeleton_min_component_length": 6,
}

EXPERIMENTS = {
    "exp1_recall_oriented": {
        "description": "Recall-oriented CA-APSRG: fill holes and closing are enabled.",
        "adaptive_morphology": {
            "enabled": True,
            "remove_small_objects": True,
            "fill_small_holes": True,
            "closing_enabled": True,
            "opening_enabled": False,
            "skeleton_guard_enabled": True,
            "small_component_area_low_density": 8,
            "small_component_area_normal": 18,
            "small_component_area_high_density": 28,
            "hole_area_low_density": 8,
            "hole_area_normal": 12,
            "hole_area_high_density": 16,
            "closing_kernel_low_density": 3,
            "closing_kernel_normal": 3,
            "closing_kernel_high_density": 5,
            "opening_kernel_low_density": 0,
            "opening_kernel_normal": 0,
            "opening_kernel_high_density": 0,
            "skeleton_restore_radius": 1,
            "skeleton_min_component_length": 6,
        },
    },
    "exp2_precision_oriented": {
        "description": "Precision-oriented CA-APSRG: aggressive small component removal and opening.",
        "adaptive_morphology": {
            "enabled": True,
            "remove_small_objects": True,
            "fill_small_holes": False,
            "closing_enabled": False,
            "opening_enabled": True,
            "skeleton_guard_enabled": False,
            "small_component_area_low_density": 12,
            "small_component_area_normal": 24,
            "small_component_area_high_density": 36,
            "hole_area_low_density": 4,
            "hole_area_normal": 8,
            "hole_area_high_density": 12,
            "closing_kernel_low_density": 0,
            "closing_kernel_normal": 0,
            "closing_kernel_high_density": 0,
            "opening_kernel_low_density": 0,
            "opening_kernel_normal": 3,
            "opening_kernel_high_density": 3,
            "skeleton_restore_radius": 1,
            "skeleton_min_component_length": 6,
        },
    },
    "exp3_balanced_main": {
        "description": "Balanced CA-APSRG main configuration.",
        "adaptive_morphology": {
            "enabled": True,
            "remove_small_objects": True,
            "fill_small_holes": False,
            "closing_enabled": False,
            "opening_enabled": False,
            "skeleton_guard_enabled": True,
            "small_component_area_low_density": 8,
            "small_component_area_normal": 18,
            "small_component_area_high_density": 28,
            "hole_area_low_density": 4,
            "hole_area_normal": 8,
            "hole_area_high_density": 12,
            **COMMON_ZERO_KERNELS,
        },
    },
    "exp4_static_morphology": {
        "description": "Static morphology ablation: small component threshold is 18/18/18.",
        "adaptive_morphology": {
            "enabled": True,
            "remove_small_objects": True,
            "fill_small_holes": False,
            "closing_enabled": False,
            "opening_enabled": False,
            "skeleton_guard_enabled": True,
            "small_component_area_low_density": 18,
            "small_component_area_normal": 18,
            "small_component_area_high_density": 18,
            "hole_area_low_density": 8,
            "hole_area_normal": 8,
            "hole_area_high_density": 8,
            **COMMON_ZERO_KERNELS,
        },
    },
    "exp5_no_skeleton_guard": {
        "description": "No skeleton guard ablation: same as balanced but skeleton guard disabled.",
        "adaptive_morphology": {
            "enabled": True,
            "remove_small_objects": True,
            "fill_small_holes": False,
            "closing_enabled": False,
            "opening_enabled": False,
            "skeleton_guard_enabled": False,
            "small_component_area_low_density": 8,
            "small_component_area_normal": 18,
            "small_component_area_high_density": 28,
            "hole_area_low_density": 4,
            "hole_area_normal": 8,
            "hole_area_high_density": 12,
            **COMMON_ZERO_KERNELS,
        },
    },
    "exp6_no_small_component": {
        "description": "No small component removal ablation.",
        "adaptive_morphology": {
            "enabled": True,
            "remove_small_objects": False,
            "fill_small_holes": False,
            "closing_enabled": False,
            "opening_enabled": False,
            "skeleton_guard_enabled": False,
            "small_component_area_low_density": 8,
            "small_component_area_normal": 18,
            "small_component_area_high_density": 28,
            "hole_area_low_density": 4,
            "hole_area_normal": 8,
            "hole_area_high_density": 12,
            **COMMON_ZERO_KERNELS,
        },
    },
}

def make_experiment_config(name: str, spec: dict) -> Path:
    cfg = deepcopy(base_config)
    cfg.setdefault("experiment", {})
    cfg["experiment"]["name"] = name
    cfg["experiment"]["description"] = spec.get("description", "")
    cfg["adaptive_morphology"] = deepcopy(spec["adaptive_morphology"])
    path = EXPERIMENT_CONFIG_DIR / f"{name}.yaml"
    save_yaml(cfg, path)
    return path

experiment_config_paths = {}
for name, spec in EXPERIMENTS.items():
    path = make_experiment_config(name, spec)
    experiment_config_paths[name] = path
    print(f"Saved {name}: {path}")


## 8. Jalankan Eksperimen

Untuk hemat waktu, Bapak dapat mengubah `EXPERIMENTS_TO_RUN`.

Contoh:
- Jalankan semua: `list(EXPERIMENTS.keys())`
- Jalankan hanya hasil utama: `["exp3_balanced_main"]`


In [ ]:
# Pilih eksperimen yang ingin dijalankan.
EXPERIMENTS_TO_RUN = list(EXPERIMENTS.keys())
# EXPERIMENTS_TO_RUN = ["exp3_balanced_main"]

CLEAN_BEFORE_RUN = True

print("Experiments to run:")
for name in EXPERIMENTS_TO_RUN:
    print("-", name)


In [ ]:
def run_one_experiment(name: str) -> None:
    config_path = experiment_config_paths[name]
    exp_output = PROJECT_DIR / "outputs" / f"experiments_{name}"
    analysis_output = PROJECT_DIR / "outputs" / f"analysis_{name}"

    if CLEAN_BEFORE_RUN:
        if exp_output.exists():
            shutil.rmtree(exp_output)
        if analysis_output.exists():
            shutil.rmtree(analysis_output)

    run_cmd([
        sys.executable,
        "scripts/03_run_batch.py",
        "--config", str(config_path),
        "--manifest", "data/manifests/manifest.csv",
        "--output-dir", str(exp_output),
    ])

    run_cmd([
        sys.executable,
        "scripts/04_summarize_results.py",
        "--config", str(config_path),
        "--results", str(exp_output / "metrics_per_image.csv"),
        "--output-dir", str(analysis_output),
    ])

for exp_name in EXPERIMENTS_TO_RUN:
    print("\n" + "=" * 80)
    print("RUNNING", exp_name)
    print("=" * 80)
    run_one_experiment(exp_name)


## 9. Kembalikan Hasil Utama ke Folder Default untuk Streamlit Cloud

Folder default yang dibaca Streamlit:

```text
outputs/experiments/
outputs/analysis/
```

Cell ini menyalin hasil **Experiment 3: Balanced CA-APSRG** ke folder default tersebut dan mengembalikan `configs/default.yaml` ke konfigurasi Experiment 3.


In [ ]:
MAIN_EXPERIMENT = "exp3_balanced_main"

main_config_path = experiment_config_paths[MAIN_EXPERIMENT]
shutil.copy2(main_config_path, BASE_CONFIG_PATH)
print("Restored default.yaml from:", main_config_path)

source_exp = PROJECT_DIR / "outputs" / f"experiments_{MAIN_EXPERIMENT}"
source_analysis = PROJECT_DIR / "outputs" / f"analysis_{MAIN_EXPERIMENT}"
target_exp = PROJECT_DIR / "outputs" / "experiments"
target_analysis = PROJECT_DIR / "outputs" / "analysis"

if target_exp.exists():
    shutil.rmtree(target_exp)
if target_analysis.exists():
    shutil.rmtree(target_analysis)

shutil.copytree(source_exp, target_exp)
shutil.copytree(source_analysis, target_analysis)

print("Copied main experiment to:")
print("-", target_exp)
print("-", target_analysis)


## 10. Buat Ringkasan Perbandingan Antar Eksperimen

Cell ini membuat tabel sederhana dari `article_table_mean_std.csv` dan `improvement_by_dataset.csv` untuk semua eksperimen yang sudah dijalankan.


In [ ]:
summary_rows = []

for name in EXPERIMENTS.keys():
    analysis_dir = PROJECT_DIR / "outputs" / f"analysis_{name}"
    improvement_path = analysis_dir / "improvement_by_dataset.csv"
    article_path = analysis_dir / "article_table_mean_std.csv"

    if not improvement_path.exists():
        continue

    imp = pd.read_csv(improvement_path)
    for _, row in imp.iterrows():
        summary_rows.append({
            "experiment": name,
            "dataset": row.get("dataset", ""),
            "n_images": row.get("n_images", None),
            "delta_precision_mean": row.get("delta_precision_mean", None),
            "delta_recall_mean": row.get("delta_recall_mean", None),
            "delta_f1_score_mean": row.get("delta_f1_score_mean", None),
            "delta_iou_mean": row.get("delta_iou_mean", None),
        })

cross_experiment_summary = pd.DataFrame(summary_rows)
cross_summary_path = PROJECT_DIR / "outputs" / "cross_experiment_summary.csv"
cross_experiment_summary.to_csv(cross_summary_path, index=False)

print("Saved:", cross_summary_path)
display(cross_experiment_summary)


## 11. Buat ZIP Hasil Eksperimen

ZIP ini dapat diunduh dari Colab atau disimpan ke Google Drive.

Secara default, ZIP akan berisi:
- `configs/default.yaml`
- `configs/experiments/*.yaml`
- `outputs/experiments_exp*/`
- `outputs/analysis_exp*/`
- `outputs/experiments/`
- `outputs/analysis/`
- `outputs/cross_experiment_summary.csv`

Jika ukuran terlalu besar, Bapak bisa membuat versi metrik saja dengan mengubah `METRICS_ONLY_ZIP = True`.


In [ ]:
METRICS_ONLY_ZIP = False

ZIP_OUTPUT_DIR = PROJECT_DIR / "outputs"
ZIP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

zip_name = "ca_apsrg_experiments_1_to_6_colab"
zip_base = ZIP_OUTPUT_DIR / zip_name
zip_path = zip_base.with_suffix(".zip")

if zip_path.exists():
    zip_path.unlink()

temp_zip_root = PROJECT_DIR / "_zip_package"
if temp_zip_root.exists():
    shutil.rmtree(temp_zip_root)
temp_zip_root.mkdir(parents=True, exist_ok=True)

# Copy configs
shutil.copytree(PROJECT_DIR / "configs", temp_zip_root / "configs")

# Copy selected outputs
out_target = temp_zip_root / "outputs"
out_target.mkdir(parents=True, exist_ok=True)

for item in (PROJECT_DIR / "outputs").iterdir():
    if item.name == zip_path.name:
        continue
    if item.name.startswith("experiments") or item.name.startswith("analysis") or item.name == "cross_experiment_summary.csv":
        src = item
        dst = out_target / item.name

        if src.is_file():
            shutil.copy2(src, dst)
            continue

        if not METRICS_ONLY_ZIP:
            shutil.copytree(src, dst)
        else:
            dst.mkdir(parents=True, exist_ok=True)
            # Copy CSV/MD/TXT only, skip PNG visual outputs.
            for file in src.rglob("*"):
                if file.is_file() and file.suffix.lower() in {".csv", ".md", ".txt", ".yaml", ".yml", ".json"}:
                    rel = file.relative_to(src)
                    (dst / rel.parent).mkdir(parents=True, exist_ok=True)
                    shutil.copy2(file, dst / rel)

archive_file = shutil.make_archive(str(zip_base), "zip", root_dir=temp_zip_root)
shutil.rmtree(temp_zip_root)

print("ZIP created:", archive_file)
print("Size MB:", Path(archive_file).stat().st_size / (1024 * 1024))


In [ ]:
# Simpan ZIP ke Google Drive
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/ca_apsrg_outputs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

drive_zip_path = DRIVE_OUTPUT_DIR / Path(archive_file).name
shutil.copy2(archive_file, drive_zip_path)

print("Copied ZIP to Google Drive:")
print(drive_zip_path)


## 12. Download ZIP dari Colab

Jalankan cell berikut jika ingin langsung download ZIP ke komputer lokal.


In [ ]:
from google.colab import files

files.download(str(zip_path))


## 13. Opsional: Jalankan Streamlit di Colab

Streamlit lebih nyaman dijalankan lokal atau di Streamlit Cloud. Di Colab bisa dijalankan dengan tunnel, tetapi tidak wajib untuk eksperimen.

Untuk Streamlit Cloud, commit folder berikut ke GitHub:
- `configs/default.yaml`
- `outputs/experiments/`
- `outputs/analysis/`
- `app.py`
- `src/ui/`
